# Project 2

Welcome to the second NLP1 project. The goal of the second tutorial was to get you familiar with using pre-trained embeddings and moving towards a compositional encoder for text classification.

Given the skills you have developed in the last week, you will now be applying them within the dataset and task you have chosen during Project 1. You will be applying hands-on the experiments you have already explored in the Tutorial 2, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and errors analyses, and tackling your own research questions. 


The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

# Get to work!

1. Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
    * Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
    * Does the model's vocabulary give enough coverage to the words in your texts? etc.
2. Explore what your chosen word embedding model captures. Some things you might consider: 
    * Is it good at representing interesting similarities (re to your choice of task)? J
    * Have you found task-specific evidence that it's better than some other reasonable alternative?
    * Do you know about limitations of this model?
    * Any limitations that you find particularly troubling for your task? 
3. Experiment with the embed-and-pool approach.  
    * What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
    * Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?
4. Experiment with a recurrent encoder (GRU or LSTM). 
    * What design choices have meaningful impact in classification performance, if any?
    * Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Did you struggle with over-fitting?
    * How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    * How did you structure you hyperparameter search?
    * Are you keeping an eye on experimental variance?
6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 
    * What analysis features did you come up with?
    * Which ones did you manage to test and how did that go?
    * What would you prioritise in the future in order to improve your systems?

Be mindful that you will be using your experiments to write up your final blog, so make sure your code is clean and neat!

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

# Tips

## Pretrained embedding models

We will post a list with some options for you. 

## Features for analysis

Features that you can interpret as an aspect of _complexity_ in light of your task's goals.

- length
- presence of rare/common words, including unknown words
- syntactic features (like parts-of-speech, syntactic complexity, syntactic voice, presence of negation, etc.)
- semantic features (eg, the average similarity of the text to training documents or some curated set of training documents, presence of words in certain semantic classes, etc.)
- task-specific features (eg, 
- model-specific features (eg, predicted label, probability assigned to predicted label, number of parameters or layers or some other model descriptor)

## Setup

In [1]:
%load_ext autoreload
%autoreload 2    

In [2]:
import nltk
import numpy as np
nltk.download('opinion_lexicon')

[nltk_data] Downloading package opinion_lexicon to
[nltk_data]     /Users/cristyrussell/nltk_data...
[nltk_data]   Package opinion_lexicon is already up-to-date!


True

In [3]:
import arviz as az

In [4]:
import sys
from pathlib import Path
root = Path.cwd().parent
sys.path.insert(0, str(root))

import part1  # we will reuse some things from last tutorial
import part2
import datasets
import string

/Users/cristyrussell/.miniforge/envs/nlp-project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# save models and outputs there to keep the directory tree organised
results_folder = Path("./results")
results_folder.mkdir(parents=True, exist_ok=True) 
results_folder

PosixPath('results')

# Dataset

In [6]:
dataset=datasets.load_dataset("gsri-18/ISEAR-dataset-complete")

In [7]:
print(dataset)


DatasetDict({
    train: Dataset({
        features: ['emotion', 'content', 'Unnamed: 2'],
        num_rows: 7516
    })
})


In [8]:
emotion_classes = part1.clslib.NamedClasses(['anger', 'disgust', 'fear', 'guilt', 'joy', 'sadness', 'shame'])
label_mapping = {name: i for i,name in enumerate(emotion_classes)}
dataset = dataset.map(lambda example: {'emotion': label_mapping[example['emotion']]})

In [9]:
corpus = dataset["train"]["content"]
len(corpus)
vocab_size = 1000
min_frequency = 2

results_folder = Path("./results")
results_folder.mkdir(parents=True, exist_ok=True)

tokenizer = part1.experiment.train_tokenizer(corpus, vocab_size, min_frequency)
tokenizer.save(f"{results_folder}/BPE_tokenizer_vs_{vocab_size}_mf_{min_frequency}.json")

#corpus_testing = dataset["test"]["content"]


## Load GloVe

In [10]:
from pathlib import Path
from urllib.request import urlopen
import shutil

def download_pretrained(option):
    """
    You can use this to download GloVe (or similar) model using wget. 
    It's fine to download it through other means, but don't unzip the file (the rest of the notebook can 
    handle the file in its zipped format).

    This function returns the path which you can use for the loader class that follows.

    The function will skip the download if it detects the file in disk. If the file in disk is corrupted, you will have to delete the file first.
    """
    if option == 'glove':
        destination = "glove.2024.wikigiga.50d.zip"
        dims = 50
        skip_first = False
    elif option == 'fasttext':
        destination = "wiki-news-300d-1M.vec.zip"
        dims = 300
        skip_first = True
    else:
        raise ValueError(f"Unkonwn option: {option}")
        
    if not Path(destination).is_file():
        if option == 'glove':
            #!wget https://nlp.stanford.edu/data/wordvecs/glove.2024.wikigiga.50d.zip
            !wget -O glove.2024.wikigiga.50d.zip 'https://surfdrive.surf.nl/s/qYfZ6PiNrMLCwpS/download'
        elif option == 'fasttext':
            !wget https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
        

    return destination, dims, skip_first

In [11]:
%%time
# depends on internet connection (but say 5 minutes or less)
embtable_version, embtable_dims, embtable_metadata = download_pretrained('glove') # in case stanford.edu is down, use fasttext (but glove is preferred)
embtable_version, embtable_dims, embtable_metadata

CPU times: user 68 μs, sys: 23 μs, total: 91 μs
Wall time: 83 μs


('glove.2024.wikigiga.50d.zip', 50, False)

In [12]:
%%time
# about 30 seconds for GloVe
trained_emb_model = part2.clslib.PretrainedEmbeddings(embtable_version, embtable_dims, skip_first_line=embtable_metadata)

CPU times: user 24.9 s, sys: 807 ms, total: 25.7 s
Wall time: 25.9 s


## Inspect GloVe

In [13]:
print("Shape of embedding matrix:", trained_emb_model.vectors.shape)
print("Size of vocabulary:", len(trained_emb_model.vocab))
print(f"UNK token {trained_emb_model.vocab.default_index}:{trained_emb_model.vocab.default_word}") 
print(f"PAD token {trained_emb_model.vocab.index(trained_emb_model.pad_token)}:{trained_emb_model.pad_token}")
print("First few entries in the vocabulary:")
for _, (i, w) in zip(range(10), trained_emb_model.vocab.items()):
    print(f"{i}:{w}")

Shape of embedding matrix: (1287615, 50)
Size of vocabulary: 1287615
UNK token 1:<unk>
PAD token 0:<pad>
First few entries in the vocabulary:
0:<pad>
1:<unk>
2:the
3:,
4:.
5:of
6:to
7:and
8:in
9:a


In [14]:
for word in ['person', 'human', 'city', 'place', 'camera-ready']:
    print(word, word in trained_emb_model, trained_emb_model.index(word), trained_emb_model.embed_word(word).shape)

person True 880 (50,)
human True 487 (50,)
city True 120 (50,)
place True 235 (50,)
camera-ready False 1 (50,)


In [15]:
trained_emb_model.embed_word('person')

array([-0.328356, -0.0617  , -1.169463,  0.029792,  0.07475 ,  0.086573,
       -0.35415 , -0.508845,  0.055547, -0.460465,  0.606028, -0.348823,
       -0.363127, -0.314696, -0.143566,  0.631562,  0.305427, -0.516298,
       -0.112355,  0.638538, -0.779977,  0.552125,  0.51446 ,  0.299915,
       -0.960752,  0.053659, -0.488988,  0.06239 ,  0.757579, -0.168628,
       -0.037003, -0.608806,  0.710988, -0.77913 , -0.320539,  0.995688,
        0.202728,  4.159241,  0.064915,  0.023952, -0.169669, -0.022096,
        0.301033,  0.629048, -0.770124,  0.066405, -0.331856,  1.177511,
        0.482992,  1.202875], dtype=float32)

Globe vocab coverage on ISEAR

In [16]:
words = []
content_col = dataset["train"]["content"]
for text in content_col:
    for punctuation in string.punctuation:
        text = text.replace(punctuation, "")
    tokens = text.lower().split()
    words.extend(tokens)

unique_words = set(words)
glove_vocab = trained_emb_model.vocab
covered_words = unique_words.intersection(glove_vocab)
coverage = len(covered_words) / len(unique_words)
coverage

0.9298620763391425

In [17]:
#Observe missing words
missing_words = list(unique_words - covered_words)
missing_words[:20]

['monthend',
 'negativei',
 'galso',
 'isul',
 'blindworm',
 'friendspartners',
 'plovdivbourgas',
 'morining',
 'mmembers',
 'diificult',
 'didcovered',
 'pittied',
 'redtapism',
 'sadistinc',
 '93010',
 'ambulants',
 'sopposed',
 'whatched',
 'putridness',
 'competitress']

## Word analogies

In [18]:
def cos_similarity(u, v):
    """
    Return the cosine similarity (a real number) of the two words in the embedding space of our model.
    """
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

def topk_ids(v, E, k=10):
    """
    word embedding: a D-dimensional vector
    E: our model's embedding matrix as a numpy array of shape [V, D]
    k: how many nearest neighbours we want to find

    Return a list with the k word ids that are nearest to the input word embedding v
     in embedding space according to cosine similarity.
    """
    temp_cos_sim = []
    temp_ids = []
    for e in range(len(E)):
        temp_cos_sim.append(cos_similarity(v, E[e]))
        temp_ids.append(e)

    order = sorted(range(len(temp_cos_sim)), key=lambda i: temp_cos_sim[i], reverse=True)
    values_sorted = [temp_cos_sim[i] for i in order]
    ids_sorted    = [temp_ids[i] for i in order]

    return ids_sorted[:10]
def analogy(a, b, c, k=10):
    """Top-k words closest to embed(a) - embed(b) + embed(c)."""
    u = (trained_emb_model.embed_word(a)
         - trained_emb_model.embed_word(b)
         + trained_emb_model.embed_word(c))
    ids = topk_ids(u, trained_emb_model.vectors, k + 3)   # a few extra, since the query words show up
    words = [trained_emb_model.word(i) for i in ids]
    return [w for w in words if w not in (a, b, c)][:k]

for a, b, c in [('king', 'man', 'woman'), ('professor', 'man', 'woman'), ('nurse', 'man', 'woman')]:
    print(f"{a} - {b} + {c}:", analogy(a, b, c))
    print(f"{a} - {c} + {b}:", analogy(a, c, b))   # flipped
    print()


/var/folders/h3/pqjwtxk96xn9s4qrwnm0g4hm0000gn/T/ipykernel_2136/1555877700.py:5: RuntimeWarning: invalid value encountered in scalar divide
  return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))


king - man + woman: ['<pad>', 'queen', 'daughter', 'throne', 'eldest', 'elizabeth', 'mother', 'princess', 'father']
king - woman + man: ['<pad>', 'reign', 'ii', 'iii', 'prince', 'ruler', 'lord', 'brother', 'iv']

professor - man + woman: ['<pad>', 'sociology', 'psychologist', 'researcher', 'ph.d.', 'university', 'psychology', 'graduated', 'lecturer']
professor - woman + man: ['<pad>', 'university', 'economics', 'associate', 'scientist', 'harvard', 'assistant', 'prof.', 'science']

nurse - man + woman: ['<pad>', 'midwife', 'pregnant', 'pediatrician', 'gynecologist', 'dentist', 'therapist', 'schoolteacher', 'obstetrician']
nurse - woman + man: ['<pad>', 'doctor', 'young', 'officer', 'trained', 'assistant', 'sergeant', 'who']



## Cosine similarities

In [19]:
def cos_sim(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

In [20]:
# Glove
happy_vector = trained_emb_model.embed_word('happy')
joyful_vector = trained_emb_model.embed_word('joyful')
angry_vector = trained_emb_model.embed_word('angry')
sad_vector = trained_emb_model.embed_word('sad')
delighted_vector = trained_emb_model.embed_word('delighted')
furious_vector = trained_emb_model.embed_word('furious')
scared_vector = trained_emb_model.embed_word('scared')
terrified_vector = trained_emb_model.embed_word('terrified')
miserable_vector = trained_emb_model.embed_word('miserable')

print("sim happy & joyful: ",cos_sim(happy_vector,joyful_vector))
print("sim happy & angry: ",cos_sim(happy_vector,angry_vector))
print("sim happy & sad: ",cos_sim(happy_vector,sad_vector))
print("sim joyful & delighted: ",cos_sim(joyful_vector,delighted_vector))
print("sim angry & furious: ",cos_sim(angry_vector,furious_vector))
print("sim scared & terrified: ",cos_sim(scared_vector,terrified_vector))
print("sim sad & miserable: ",cos_sim(sad_vector,miserable_vector))

sim happy & joyful:  0.6150653
sim happy & angry:  0.6163013
sim happy & sad:  0.74169546
sim joyful & delighted:  0.5208035
sim angry & furious:  0.86292577
sim scared & terrified:  0.8134786
sim sad & miserable:  0.71245104


In [21]:
from urllib.request import urlretrieve

url ="https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip"
filename = "wiki-news-300d-1M.vec.zip"
urlretrieve(url, filename)

fasttext_model = part2.clslib.PretrainedEmbeddings(embtable_version,embtable_dims, embtable_metadata, skip_first_line=embtable_metadata)

In [22]:
unique_words = set(words)
fasttext_vocab = fasttext_model.vocab
covered_words = unique_words.intersection(fasttext_vocab)
coverage = len(covered_words) / len(unique_words)
coverage

0.9298620763391425

In [23]:
fasttext_happy_vector = fasttext_model.embed_word('happy')
fasttext_joyful_vector = fasttext_model.embed_word('joyful')
fasttext_angry_vector = fasttext_model.embed_word('angry')
fasttext_sad_vector = fasttext_model.embed_word('sad')
fasttext_delighted_vector = fasttext_model.embed_word('delighted')
fasttext_furious_vector = fasttext_model.embed_word('furious')
fasttext_scared_vector = fasttext_model.embed_word('scared')
fasttext_terrified_vector = fasttext_model.embed_word('terrified')
fasttext_miserable_vector = fasttext_model.embed_word('miserable')
print("sim happy & joyful: ",cos_sim(fasttext_happy_vector,fasttext_joyful_vector))
print("sim happy & angry: ",cos_sim(fasttext_happy_vector,fasttext_angry_vector))
print("sim happy & sad: ",cos_sim(fasttext_happy_vector,fasttext_sad_vector))
print("sim joyful & delighted: ",cos_sim(fasttext_joyful_vector,fasttext_delighted_vector))
print("sim angry & furious: ",cos_sim(fasttext_angry_vector,fasttext_furious_vector))
print("sim scared & terrified: ",cos_sim(fasttext_scared_vector,fasttext_terrified_vector))
print("sim sad & miserable: ",cos_sim(fasttext_sad_vector,fasttext_miserable_vector))

sim happy & joyful:  0.6150653
sim happy & angry:  0.6163013
sim happy & sad:  0.74169546
sim joyful & delighted:  0.5208035
sim angry & furious:  0.86292577
sim scared & terrified:  0.8134786
sim sad & miserable:  0.71245104


## Tokenizers

In [24]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/cristyrussell/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/cristyrussell/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [25]:
from tabulate import tabulate

demo_texts = ['Dog bites man.', 'This tutorial is offered as part of NLP1, an MSc level course in the AI programme of the UvA.']
tokenizer_for_bow = part2.clslib.BasicTokenizer(
    lowercase=True, 
    remove_punct=True, 
    remove_stopwords=True, 
    language='english',
    vocab=trained_emb_model.vocab
)

for text in demo_texts:
    print(text)
    print(tabulate(list(zip(tokenizer_for_bow.ids(text), tokenizer_for_bow.tokens(text))), headers=['id', 'token']))    
    print()

Dog bites man.
   id  token
-----  -------
 3039  dog
16342  bites
  311  man

This tutorial is offered as part of NLP1, an MSc level course in the AI programme of the UvA.
   id  token
-----  ---------
42158  tutorial
 1123  offered
  157  part
    1  nlp1
24596  msc
  502  level
  760  course
 7388  ai
 3024  programme
45065  uva



In [26]:
tokenizer_for_rnn = part2.clslib.BasicTokenizer(
    lowercase=True, 
    remove_punct=False,
    remove_stopwords=False, 
    language='english',
    vocab=trained_emb_model.vocab
)

for text in demo_texts:
    print(text)
    print(tabulate(list(zip(tokenizer_for_rnn.ids(text), tokenizer_for_rnn.tokens(text))), headers=['id', 'token']))    
    print()

Dog bites man.
   id  token
-----  -------
 3039  dog
16342  bites
  311  man
    4  .

This tutorial is offered as part of NLP1, an MSc level course in the AI programme of the UvA.
   id  token
-----  ---------
   41  this
42158  tutorial
   17  is
 1123  offered
   21  as
  157  part
    5  of
    1  nlp1
    3  ,
   31  an
24596  msc
  502  level
  760  course
    8  in
    2  the
 7388  ai
 3024  programme
    5  of
    2  the
45065  uva
    4  .



## Embed-and-pool

In [27]:
my_seeds = [0, 3, 5]
my_seeds = [0] # to add less seeds which saves comp time

num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

for seed in my_seeds:
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(dataset["train"]["content"], dataset["train"]["emotion"])),
            k=num_folds,
            seed=seed
        )
    )

my_classifiers = OrderedDict()

NameError: name 'OrderedDict' is not defined

In [ ]:
%%time
my_classifiers['embed-and-pool'] = []
for run, seed in enumerate(my_seeds):   # about 20 minutes for a full run (with 5 folds)
    run_folder = f"{results_folder}/embed-and-pool/seed-{seed}"
    my_classifiers['embed-and-pool'].append(
        part2.experiment.cross_validation(
            outcome_space=emotion_classes,
            data_splits=my_cross_validation_sets[run],
            tokenizer=tokenizer_for_bow,
            emb_model=trained_emb_model,
            enc_type='embed-and-pool', 
            enc_kwargs={'freeze_embeddings': True},
            regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.5},
            opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 20},    
            device='cpu', #'cpu'
            save_to=run_folder,        
        )
    )

## Results container

In [ ]:
from collections import OrderedDict
my_classifiers = OrderedDict()

## Plot CV results

In [ ]:
part1.analysis.plot_box_and_points(part1.analysis.make_data_frame(my_classifiers))

## Load trained models

In [ ]:
em00 = part2.experiment.load(f"results/embed-and-pool/seed-0/fold-0/model", trained_emb_model)

In [ ]:
em00_dt_logprobs = part2.experiment.predict_logprobs(em00, tokenizer_for_bow, [text for text, label in my_test_sets[0]['devtest']], 64, 'cpu')
em00_dt_logprobs.shape

In [ ]:
from sklearn.metrics import classification_report

## Error analysis features

In [ ]:
# we will be using textstat as a way to otain some descriptive features of text; 
# we encourage you to look for things that are meaningful to your own task/dataset
# this is just to demo the kind of thing you can do 
import textstat  

import pandas as pd

table = []
systems = dict()
items = dict()

# each model uses a certain tokenizer, and we don't want to accidentally use the wrong one
name_to_tokenizer = {
    'embed-and-pool': tokenizer_for_bow, 
    'lstm': tokenizer_for_rnn
}

for model_family, model_data in my_classifiers.items(): # for all our trained classifiers
    for run, (seed, run_data) in enumerate(zip(my_seeds, model_data)):
        # we get the devtest set that we kep aside for each run
        # both texts and labels
        run_texts = [text for text, label in my_test_sets[run]['devtest']]
        run_labels = np.array([label for text, label in my_test_sets[run]['devtest']], dtype=int)
        
        for fold in range(num_folds):            
            # load our model
            trained_nn = part2.experiment.load(f"results/{model_family}/seed-{seed}/fold-{fold}/model", trained_emb_model)
            # test it on the devtest
            logprobs = part2.experiment.predict_logprobs(
                trained_nn, 
                name_to_tokenizer[model_family], 
                run_texts, 
                64, 
                'cpu'
            )
            preds = np.argmax(logprobs, -1)
    
            # then featurize the texts and compute the correctness flags
            for i in range(len(preds)):                
                                
                # find an int id for each system
                sid = systems.get(model_family, None)                
                if sid is None:
                    sid = len(systems)
                    systems[model_family] = sid
                # find an int id for each item
                item = my_test_sets[run]['devtest_ids'][i]                  
                item_id = items.get(item, None)                
                if item_id is None:
                    item_id = len(items)
                    items[item] = item_id

                # text, pred, and label
                x = run_texts[i]
                y_ = preds[i]
                y = run_labels[i]

                # put it all in a table, along with some features
                table.append([
                    model_family, 
                    item,
                    sid, 
                    item_id, 
                    run, 
                    fold, 
                    y, 
                    y_, 
                    int(y == y_), # correctness flag
                    # some features
                    textstat.lexicon_count(x),
                    textstat.flesch_reading_ease(x),
                    textstat.reading_time(x),
                    textstat.flesch_kincaid_grade(x)
                ])

df = pd.DataFrame(table, columns=['system', 'item', 'system_id', 'item_id', 'run', 'fold',  'label', 'pred', 'correct', 'lexicon_count', 'flesch_reading_ease', 'reading_time', 'flesch_kincaid_grade'])
df

## Posterior samples

In [ ]:
posterior = mcmc.get_samples()
print("Weight samples:", posterior["w"].shape)
print("System shift samples:", posterior["sys_shift"].shape)
print("Divergences:", int(mcmc.get_extra_fields()["diverging"].sum()))

## Posterior plots

In [ ]:
az_stats_params = {'point_estimate': 'median', 'ci_kind': 'hdi', 'ci_probs': (0.5, 0.95)}

In [ ]:
az.plot_forest(
    posterior_data, 
    var_names=['sys_shift'],    
    combined=True,           # Combine chains
    **az_stats_params
)

In [ ]:
az.plot_forest(
    posterior_data, 
    var_names=['accuracy'],    
    combined=True,           # Combine chains
    **az_stats_params
)

In [ ]:
az.plot_forest(
    posterior_data, 
    var_names=['w'],    
    combined=True,           # Combine chains    
    shade_label="system",
    figure_kwargs={
        "figsize": (16, 8),
        "width_ratios": [2, 3],  # Label panel : interval panel
        "layout": "none",
    },
    **az_stats_params,
)

In [ ]:
# Create a plotting copy containing only w, with reordered dimensions.
plot_data = posterior_data.copy()
plot_data["posterior"] = (
    posterior_data.posterior["w"]
    .transpose("chain", "draw", "feature", "system")
    .to_dataset()
)

az.plot_forest(
    plot_data,
    var_names=["w"],
    combined=True,
    shade_label="feature",
    figure_kwargs={
        "figsize": (16, 8),
        "width_ratios": [2, 3],
        "layout": "none",
    },
    **az_stats_params,
)